In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from transformers import AutoProcessor
import os
from vla.constants import PROJECT_ROOT
from vla.models.smolvla import smolvla

In [ ]:
def get_multi_layer_attention(model, pil_image, task_description, device="cuda"):
    """
    Hooks into multiple percentiles of the LLM to track attention evolution.
    """
    attention_maps = {}

    # Language Model
    hf_vlm = model.model.vlm_with_expert.vlm
    llm = hf_vlm.model.text_model
    total_layers = len(llm.layers)
    print(f"Isolated the Language Model successfully. Total layers: {total_layers}")

    # layer choice
    layer_indices = [18,19,20,21,22,23,24]
    print(f"Targeting layer indices: {layer_indices}")

    # Create a unique hook for each layer
    def get_forward_hook(layer_idx):
        def hook(module, input, output):
            print(f"Hook triggered for layer {layer_idx}!")
            # output[1] contains the attention weights
            attention_maps[layer_idx] = output[1].detach().cpu()

        return hook

    # Register hooks on all target layers
    hook_handles = []
    for idx in layer_indices:
        target_layer = llm.layers[idx].self_attn
        handle = target_layer.register_forward_hook(get_forward_hook(idx))
        hook_handles.append(handle)

    # Force the LLM to output attention
    llm.config._attn_implementation = "eager"
    llm.config.output_attentions = True

    # Process the Image and Text together
    print("Processing image and text prompt...")
    # FIX: Use the processor from the CHECKPOINT, not the base model.
    try:
        processor = AutoProcessor.from_pretrained("HuggingFaceVLA/smolvla_libero")
    except Exception:
        print("Could not load processor from checkpoint, falling back to base.")
        processor = AutoProcessor.from_pretrained("HuggingFaceTB/SmolVLM2-500M-Instruct")

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": task_description}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt, images=pil_image, return_tensors="pt").to(device)

    # Check input resolution
    print(f"Input image shape: {inputs['pixel_values'].shape}")

    # Ensure image dtype matches vision model
    vision_dtype = next(hf_vlm.model.vision_model.parameters()).dtype
    inputs["pixel_values"] = inputs["pixel_values"].to(vision_dtype)

    # Run forward pass on the FULL VLM
    print("Running forward pass through the VLM...")
    with torch.no_grad():
        _ = hf_vlm(**inputs)

    # Clean up hooks
    for handle in hook_handles:
        handle.remove()

    return attention_maps, inputs, layer_indices, processor

In [ ]:
import scipy.ndimage as ndimage


def visualize_multi_layer_heatmap(attention_maps_dict, inputs, pil_image, layer_indices, processor, target_word):
    """
    Improved version: Addresses Attention Sinks and Pixel Shuffle artifacts.
    SmolVLM2 layout: 16 local crops (row_X_col_Y) followed by 1 global crop — 17 total.
    The global crop is always the LAST set of 64 image tokens.
    """
    # Find the target token index
    input_ids = inputs["input_ids"][0].tolist()
    tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)

    target_idx = -1
    for i, t in enumerate(tokens):
        if target_word.lower() in t.lower():
            target_idx = i
            print(f"Tracking attention for word '{t}' (token index {i})")
            break

    if target_idx == -1:
        print(f"Warning: Could not find '{target_word}'. Defaulting to the last token.")
        target_idx = -1

    # Find image token indices
    most_common_token = Counter(input_ids).most_common(1)[0][0]
    image_token_indices = [i for i, token in enumerate(input_ids) if token == most_common_token]
    num_crops = inputs["pixel_values"].shape[1]
    tokens_per_crop = len(image_token_indices) // num_crops  # always 64

    # SmolVLM2: global view is the LAST crop (after all local crops) - fixed from assumption of it being the first
    global_token_indices = image_token_indices[-tokens_per_crop:]

    # Setup the plot
    fig, axes = plt.subplots(1, len(layer_indices) + 1, figsize=(24, 4))

    axes[0].imshow(pil_image)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Process and plot each layer
    for plot_idx, layer_idx in enumerate(layer_indices):
        # Average across attention heads
        attn = attention_maps_dict[layer_idx].squeeze(0).mean(dim=0)
        word_attn = attn[target_idx, :]

        # Extract attention over the global crop only (64 tokens)
        global_crop_attn = word_attn[global_token_indices]

        # Reshape to compressed 2D grid (8x8)
        grid_size = int(np.sqrt(tokens_per_crop))  # 8
        salience_grid = global_crop_attn.reshape(grid_size, grid_size)

        # Convert bfloat16 to float32 -- fixed stupid issue :/
        if salience_grid.dtype == torch.bfloat16:
            salience_grid = salience_grid.to(torch.float32)
        salience_np = salience_grid.numpy()

        # clipping quite aggresviely to actually se the focus instead of just attention dumps
        vmax = np.percentile(salience_np, 80)
        vmin = np.percentile(salience_np, 2)
        salience_np = np.clip(salience_np, vmin, vmax)

        # Normalize (now safe from extreme outliers!!!!!)
        salience_np = (salience_np - salience_np.min()) / (salience_np.max() - salience_np.min() + 1e-8)

        #Beautifying additions:
        # Pixel Shuffle Smoothing
        # Reconstructs the semantic spatial spread before the plot scales it up
        # salience_np = ndimage.gaussian_filter(salience_np, sigma=0.8)

        # Plot with BICUBIC interpolation for final smoothness
        ax = axes[plot_idx + 1]
        ax.imshow(pil_image)
        im = ax.imshow(salience_np, cmap='jet', alpha=0.5, extent=[0, pil_image.width, pil_image.height, 0]) # can also add some beautifying parameters here so its not just a pixelated mess
        ax.set_title(f"Layer {layer_idx}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
checkpoint = "HuggingFaceVLA/smolvla_libero"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model from checkpoint: {checkpoint} on device: {device}")
model, model_id, action_dim = smolvla(checkpoint, device)

In [ ]:
# Choose suite -- only need to define suite name and a word in the task, rest is automatic
suite_name = "object"
# Pick a specific word to track (e.g., "bowl" or "box")
target_word = "juice"

task_description = os.listdir(PROJECT_ROOT / "data/images/libero" / suite_name)[0]

# Load Image
img_path = PROJECT_ROOT / "data/images/libero" / suite_name / "ep0000_task_0" / "frame0000.png"
image = Image.open(img_path).convert("RGB")

# Load Full Task Description
task_path = (
    PROJECT_ROOT / "data/images/libero" / suite_name / task_description / "task.txt"
)

with open(task_path) as f:
    task_description = f.read().strip()

print(f"Task: {task_description}")
print(f"Task path: {task_path}")

In [ ]:
attention_maps_dict, model_inputs, layer_indices, processor = get_multi_layer_attention(
    model, image, task_description, device
)

In [ ]:
print("Generating multi-layer heatmap overlay...")
visualize_multi_layer_heatmap(attention_maps_dict, model_inputs, image, layer_indices, processor, target_word)

# Run through all words:

In [ ]:
for word in task_description.split():
    focus_word = word
    print(f"Generating heatmap for word: {focus_word}")
    attention_maps_dict, model_inputs, layer_indices, processor = get_multi_layer_attention(
        model, image, task_description, device
    )
    print("Generating multi-layer heatmap overlay...")
    visualize_multi_layer_heatmap(attention_maps_dict, model_inputs, image, layer_indices, processor, focus_word)